# PatchCore FAISS Optimization Benchmark (IVF, HNSW, PQ)

이 노트북은 PatchCore의 kNN 탐색 병목을 해결하기 위해 FAISS의 최적화 인덱스를 적용하고 비교 분석합니다.

In [ ]:
# 1. 환경 설정 및 코드 클론
import os
if not os.path.exists('patchcore-inspection'):
    !git clone https://github.com/amazon-science/patchcore-inspection.git

%cd patchcore-inspection

# 2. 필수 라이브러리 설치
!pip install -q faiss-gpu-cu12
!pip install -q click pretrainedmodels scikit-image scikit-learn scipy tqdm psutil pandas

import sys
sys.path.append('src')

# 3. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import faiss
import numpy as np
import torch
import time
import gc
import pandas as pd
from patchcore.common import FaissNN

class FaissIVF(FaissNN):
    def __init__(self, nlist=128, nprobe=10, on_gpu=False, num_workers=4):
        super().__init__(on_gpu, num_workers)
        self.nlist = nlist
        self.nprobe = nprobe
    def _create_index(self, dimension):
        quantizer = faiss.IndexFlatL2(dimension)
        index = faiss.IndexIVFFlat(quantizer, dimension, self.nlist, faiss.METRIC_L2)
        return self._index_to_gpu(index)
    def _train(self, index, features):
        if not index.is_trained: index.train(features)
        index.nprobe = self.nprobe

class FaissHNSW(FaissNN):
    def __init__(self, M=32, efSearch=64, on_gpu=False, num_workers=4):
        # HNSW is CPU-only in standard Faiss cloner
        super().__init__(on_gpu=False, num_workers=num_workers)
        self.M = M
        self.efSearch = efSearch
    def _create_index(self, dimension):
        index = faiss.IndexHNSWFlat(dimension, self.M)
        index.hnsw.efSearch = self.efSearch
        return index

class FaissPQ(FaissNN):
    def __init__(self, m=32, nbits=8, on_gpu=False, num_workers=4):
        # Standalone PQ is better handled on CPU for stability
        super().__init__(on_gpu=False, num_workers=num_workers)
        self.m = m
        self.nbits = nbits
    def _create_index(self, dimension):
        index = faiss.IndexPQ(dimension, self.m, self.nbits)
        return index
    def _train(self, index, features):
        if not index.is_trained: index.train(features)

In [ ]:
import patchcore.backbones
import patchcore.common
import patchcore.metrics
import patchcore.patchcore
import patchcore.sampler
import patchcore.datasets.mvtec as mvtec

MVTEC_PATH = "/content/drive/MyDrive/mvtec" # 구글 드라이브 경로 확인

def run_benchmark(category, index_class, index_name, **index_kwargs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"--- Testing {index_name} on {category} ---")
    
    # 1. Dataloader
    train_ds = mvtec.MVTecDataset(MVTEC_PATH, classname=category, split=mvtec.DatasetSplit.TRAIN)
    test_ds = mvtec.MVTecDataset(MVTEC_PATH, classname=category, split=mvtec.DatasetSplit.TEST)
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=2, shuffle=False, num_workers=4)
    test_dl = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4)
    
    # 2. Sampler 객체 생성 (튜플 방지)
    sampler_obj = patchcore.sampler.ApproximateGreedyCoresetSampler(percentage=0.1, device=device)
    print(f"Sampler type: {type(sampler_obj)}")
    
    # 3. PatchCore Setup
    backbone = patchcore.backbones.load("wideresnet50")
    nn_method = index_class(on_gpu=True, **index_kwargs)
    
    model = patchcore.patchcore.PatchCore(device)
    model.load(
        backbone=backbone,
        layers_to_extract_from=["layer2", "layer3"],
        device=device,
        input_shape=(3, 224, 224),
        pretrain_embed_dimension=1024,
        target_embed_dimension=1024,
        patchsize=3,
        featuresampler=sampler_obj, # 명시적으로 객체 전달
        nn_method=nn_method
    )
    
    # 4. Training (Fit)
    t_fit_start = time.perf_counter()
    model.fit(train_dl)
    fit_time = time.perf_counter() - t_fit_start
    print(f"Fit time: {fit_time:.2f}s")
    
    # 5. Inference Latency
    latencies = []
    for i, data in enumerate(test_dl):
        img = data["image"].to(device)
        t0 = time.perf_counter()
        _ = model.predict(img)
        latencies.append((time.perf_counter() - t0) * 1000)
        if i >= 29: break # 시간 절약을 위해 30개만 테스트
        
    avg_latency = np.mean(latencies)
    print(f"Avg Latency: {avg_latency:.2f}ms")
    
    return {
        "Category": category,
        "Method": index_name,
        "Fit Time (s)": f"{fit_time:.2f}",
        "Inference (ms)": f"{avg_latency:.2f}"
    }

# 실험 실행
results = []
categories = ["bottle", "hazelnut"]
methods = [
    (patchcore.common.FaissNN, "FlatL2", {}),
    (FaissIVF, "IVF-128", {"nlist": 128, "nprobe": 10}),
    (FaissHNSW, "HNSW-32", {"M": 32, "efSearch": 64}),
    (FaissPQ, "PQ-32", {"m": 32, "nbits": 8})
]

for cat in categories:
    for cls, name, kwargs in methods:
        try:
            res = run_benchmark(cat, cls, name, **kwargs)
            results.append(res)
        except Exception as e:
            print(f"Error in {name}: {e}")
            continue

pd.DataFrame(results)